# sources

> Add concepts to the concept graph from any text, from a vault document and from vedicreader.

This module adds concepts to the `learning` shelf. A model finds the core ideas in a text, and code merges them with the concepts on the shelf.
The vedicreader import reads the repo files and needs no model.
The tutor then asks about the concepts that this module adds.

In [ ]:
#| default_exp sources

In [ ]:
#| hide
from fastcore.test import *

In [ ]:
#| hide
import os, tempfile
os.environ['VISHALAKSHI_VAULT'] = os.environ['ABHYASA_VAULT'] = os.path.join(tempfile.mkdtemp(), 'vault.db')   # a test that opens the default vault gets a temporary one

These are the imports of the module.

In [ ]:
#| export
import json, os, re
from pathlib import Path
from fastcore.utils import *
from abhyasa.store import Learning, slugify, _union
from abhyasa.llm import ask_json
from abhyasa.prompts import register, prompt

## Extraction from text

The model lists the core ideas of a text. Each idea has a true, self-contained statement and its prereqs.
Code does the merge. When a slug or an alias matches a concept on the shelf, it is the same concept.
The lists of that concept get the new items, and its learner state does not change.

`EXTRACT_SP` and `extract_prompt` ask the model for the concepts. The model also gives the section, one quote, the figure, the ways that suit the idea, and the people and dates. `EXTRACT_SP` is a registered prompt, so you can change it on the Settings page.

In [ ]:
#| export
EXTRACT_SP = register('extract', 'Concept finder', 'How the tutor finds the ideas in a text that you add.', """You find the core ideas in a text, for a learner who will be tested on them.
- A concept is one idea a learner can be asked about: a definition, a rule, a method, a result, a distinction.
- `statement` is true and self-contained: it makes sense without the text. One to three sentences. Maths in $...$.
- Skip trivia, examples and names that are not ideas. Prefer 4-15 concepts for a page or two.
- `prereqs` names other concepts (from your list or from the known list) that a learner must know first.
- List the concepts in the order of the text.
- `section` is the title of the part of the text that holds the idea.
- `quote` is one short sentence of the text, copied exactly.
- `figure` is the link of one image of the text that shows the idea, copied exactly. Use "" when there is none.
- `ways` names the ways that suit the idea. Use only names from the list in the request.
- `moves` names the things that change in the idea, for example "weights". Use [] when nothing changes.
- `when` is the year of the idea, or null. `who` names the people of the idea.""")

def extract_prompt(text, subject='', known=()):
    from abhyasa.method import METHODS
    kn = ', '.join(known[:150])
    return ((f"Subject: {subject}\n" if subject else '') + (f"Concepts the learner already has: {kn}\n" if kn else '')
            + f"Ways: {', '.join(METHODS)}\n\nText:\n{text}\n\n"
            'Return JSON: {"concepts": [{"name": str, "statement": str, "aliases": [str], "prereqs": [str], "section": str, "quote": str, '
            '"figure": str, "ways": [str], "moves": [str], "when": int|null, "who": [str]}]}')

`_index` finds the slug for a name or an alias. `_windows` cuts a long text at paragraph breaks. `doc_pages` reads every chunk of a vault document in page order. It adds 1 to the page, because the vault counts pages from 0 and a reader counts from 1.

In [ ]:
#| export
def _index(lr):
    "Map each lower-case name, alias and slug on the shelf to its slug."
    out = {}
    for c in lr.concepts():
        for n in (c.slug, c.name, *(c.aliases or ())): out[n.lower()] = c.slug
    return out

def _windows(text, size=12000):
    "Cut a long text at paragraph breaks into windows of about `size` characters."
    out, cur = [], ''
    for p in re.split(r'\n\s*\n', text):
        if cur and len(cur) + len(p) > size: out.append(cur); cur = ''
        cur += p + '\n\n'
    return [*out, cur] if cur.strip() else out

`doc_pages` returns the chunks of a document.

In [ ]:
#| export
def doc_pages(lr:Learning, ref:str, shelf:str='store') -> AttrDict:
    "Every chunk of one vault document in page order, as `(page, text)` with the page a reader sees (from 1)."
    sh = lr.vault if shelf == 'store' else lr.vault.shelf(shelf)
    if (d := sh.doc(ref)) is None: raise ValueError(f'no document in the vault matching {ref!r}')
    did = d['id'].replace("'", "''")
    cs = sorted(sh.store(where=f"doc_id='{did}'", select='content, page, rowid as rowid'), key=lambda c: (-1 if c['page'] is None else c['page'], c['rowid']))
    return AttrDict(doc_id=d['id'], title=d['title'], source=d['source'], pages=[(None if c['page'] is None else c['page'] + 1, c['content']) for c in cs])

`locate` finds a quote in the pages. It folds white space and case, and it looks in all pages, so a window border does not lose a quote.

In [ ]:
#| export
def _fold(s):
    "`s` in lower case with each run of white space made one space, and the place in `s` of each character."
    out, at = [], []
    for i, ch in enumerate(s):
        if ch.isspace():
            if out and out[-1] != ' ': out.append(' '); at.append(i)
        else:
            for x in ch.lower(): out.append(x); at.append(i)
    return ''.join(out), at

def _locate(pages, quote):
    "`(index of the chunk, start, end)` of `quote` in `pages`, or None."
    q = _fold((quote or '').strip())[0]
    if len(q) < 12: return None
    for i, (_, t) in enumerate(pages):
        f, at = _fold(t)
        if (j := f.find(q)) >= 0: return i, at[j], at[j + len(q) - 1] + 1

def locate(pages:list, quote:str):
    "`(page, start, end)` of `quote` in the `(page, text)` list `pages`, with white space and case folded. None when it is not there or too short."
    r = _locate(pages, quote)
    return r and (pages[r[0]][0], r[1], r[2])

`figure_path` keeps a figure only when the file is in the assets folder of the document and is an image.

In [ ]:
#| export
IMG_SUFFIXES = ('.png', '.jpg', '.jpeg', '.gif', '.webp')

def figure_path(lr:Learning, source:str, link:str):
    "The path of the image `link` of the document `source`, from the assets folder. None when the file is not there, is outside that folder or is not an image."
    if not (source and link): return None
    root = lr.vault.assets().resolve()
    try: p = (lr.vault.assets(Path(source).stem)/str(link)).resolve()
    except (ValueError, OSError): return None
    if p.suffix.lower() in IMG_SUFFIXES and p.is_file() and p.is_relative_to(root): return str(p.relative_to(root))

`placement` gives the entry of a concept for one document.

In [ ]:
#| export
def placement(c, doc_id:str):
    "The newest provenance entry of the concept `c` with this `doc_id`, or None."
    return next((AttrDict(p) for p in reversed(c.get('provenance') or ()) if p.get('doc_id') == doc_id), None)

`extract` adds the concepts of a text. When it gets pages, it also keeps where each idea is in the text: the order, the section, the page and the passage. `_read` asks the model for each window. `_places` puts the ideas in the order of the text. `from_vault` does the same for one document of the vault.

In [ ]:
#| export
def _strs(x): return [s for s in x if isinstance(s, str)] if isinstance(x, list) else []

def _doc_source(lr, source):
    "The source path of the vault document that `source` names, or ''."
    if not (source and source.get('doc_id')): return ''
    sh = lr.vault if source.get('shelf', 'store') == 'store' else lr.vault.shelf(source['shelf'])
    return (sh.doc(source['doc_id']) or {}).get('source') or ''

def _read(lr, ask, text, subject, pages):
    "Ask for the concepts of each window of the text, and return `(slug, concept, prereq slugs)` in the order of the replies."
    idx, out = _index(lr), []
    for w in _windows('\n\n'.join(t for _, t in pages)):
        r = ask_json(ask, extract_prompt(w, subject, sorted({v for v in idx.values()})), prompt('extract'))
        cs = r.get('concepts', []) if isinstance(r, dict) else r
        slug_of = {}
        for c in cs:
            names = [c['name'], *(c.get('aliases') or ())]
            slug_of[c['name'].lower()] = next((idx[n.lower()] for n in names if n.lower() in idx), slugify(c['name']))
        for c in cs:
            s = slug_of[c['name'].lower()]
            out.append((s, c, [slug_of.get(p.lower()) or idx.get(p.lower()) for p in c.get('prereqs') or ()]))
        for c in cs: idx.update({n.lower(): slug_of[c['name'].lower()] for n in (c['name'], *(c.get('aliases') or ()))})
    return out

def _places(pages, recs):
    "The place of each concept in the text: a list of `(order, located)`. A concept that is not found follows the one before it."
    locs, key, keys = [_locate(pages, c.get('quote')) for _, c, _ in recs], (-1, -1), []
    for l in locs: key = l[:2] if l else key; keys.append(key)
    rank = {i: n for n, i in enumerate(sorted(range(len(recs)), key=keys.__getitem__))}
    return [(rank[i], l) for i, l in enumerate(locs)]

def extract(lr:Learning, ask, text:str='', subject:str='', source:dict=None, pages:list=None) -> L:
    "Add the core ideas of `text` (or of `pages`, a list of `(page, text)`) to the shelf as concepts, merged with the concepts there. Return the slugs that it changed."
    pgs = pages if pages is not None else [(None, text)]
    recs = _read(lr, ask, text, subject, pgs)
    places = _places(pgs, recs) if pages is not None else [(None, None)] * len(recs)
    src = _doc_source(lr, source) if pages is not None else ''
    touched = L()
    for (s, c, pre), (order, loc) in zip(recs, places):
        old, fig = lr.concept(s) or {}, figure_path(lr, src, c.get('figure'))
        moves = _union(old.get('moves'), _strs(c.get('moves')))
        meta = dict(ways=_union(old.get('ways'), _strs(c.get('ways'))), moves=moves, who=_union(old.get('who'), _strs(c.get('who'))),
                    when=c.get('when', old.get('when')), image=fig or old.get('image'), visual=bool(moves))
        prov = [source] if source else []
        if pages is not None and source is not None:
            t = pgs[loc[0]][1][loc[1]:loc[2]][:600] if loc else ''
            prov = [{**source, 'order': order, 'section': c.get('section') or '', 'page': pgs[loc[0]][0] if loc else None,
                     'span': [loc[1], loc[1] + len(t)] if loc else None, 'passage': t, 'figure': fig}]
        lr.put(c['name'], c.get('statement', ''), subject, slug=s, aliases=c.get('aliases') or (), prereqs=[p for p in pre if p], provenance=prov, **meta)
        touched.append(s)
    return touched.unique()

def from_vault(lr:Learning, ask, ref:str, subject:str='', shelf:str='store') -> L:
    "Extract concepts from one document of a shelf of the vault, by doc id, source or title."
    d = doc_pages(lr, ref, shelf)
    return extract(lr, ask, subject=subject, source=dict(src='vault', shelf=shelf, doc_id=d.doc_id, title=d.title), pages=d.pages)

A fake model returns three concepts. One matches `Ranking` on the shelf by its alias `rank`.

In [ ]:
from abhyasa.store import temp_learning
from abhyasa.llm import Fake
lr = temp_learning()
lr.put('Ranking', 'Order items by a score.', 'ml', aliases=['rank'])
fake = Fake({'Return JSON: {"concepts"': dict(concepts=[
    dict(name='Reciprocal rank fusion', statement='Sum 1/(k + rank) over lists.', aliases=['RRF'], prereqs=['rank', 'Ranked list']),
    dict(name='Ranked list', statement='Items in score order.', aliases=[], prereqs=[]),
    dict(name='Ranking', statement='Ordering by score.', aliases=['ordering'], prereqs=[])])})

`extract` returns the changed slugs. The prereqs become slugs, and the old concept gets the new alias and statement.

In [ ]:
test_eq(sorted(extract(lr, fake, 'some text', 'ml', dict(src='test'))), ['ranked-list', 'ranking', 'reciprocal-rank-fusion'])
test_eq(lr.concept('reciprocal-rank-fusion').prereqs, ['ranking', 'ranked-list'])
test_eq(lr.concept('ranking').aliases, ['rank', 'ordering'])
test_eq(lr.concept('ranking').statement, 'Ordering by score.')

`_windows` cuts at paragraph breaks.

In [ ]:
test_eq(len(_windows('a\n\n' * 10, size=5)), 5)

## Reading order and where each idea stands

When the text comes as pages, `extract` also keeps the order, the section, the page and the passage of each idea. These tests use a small paper in a temporary vault. The paper data is in `tests/walk_data.py`.

In [ ]:
#| hide
import sys; sys.path.insert(0, '../tests'); from walk_data import *
from abhyasa.prompts import PROMPTS

A fake model names three ideas of the paper. `from_vault` returns them in the order of the text.

In [ ]:
lr = temp_learning(); did = add_paper(lr)
ask = Fake({'Return JSON: {"concepts"': EXTRACT_REPLY})
slugs = from_vault(lr, ask, did)
test_eq(list(slugs), ['sequential-computation', 'softmax', 'scaling-by-the-root-of-dk'])

The placement of an idea is in its provenance. The page is the page a reader sees: the vault counts pages from 0, so the stored page 3 is page 4.

In [ ]:
p = placement(lr.concept('softmax'), did)
test_eq((p.order, p.section, p.page), (1, '3.2.1 Scaled Dot-Product Attention', 4))
assert p.passage.startswith('apply a softmax function')
test_eq(p.figure, 'attention/images/fig2.png')

The way fields, the picture and the people go on the concept. The prompt has a label for the Settings page.

In [ ]:
test_eq(lr.concept('softmax').ways, ['code', 'maths'])
test_eq(lr.concept('softmax').visual, True)
test_eq(lr.concept('softmax').image, 'attention/images/fig2.png')
test_eq(lr.concept('scaling-by-the-root-of-dk').who, ['Vaswani'])
test_eq(PROMPTS['extract'].label, 'Concept finder')

A second document that names Softmax keeps its own placement. The ways of the two documents merge.

In [ ]:
d2 = lr.vault.add([(0, 'Softmax gives weights.')], 'Notes', source='/tmp/notes.md', kind='file')['doc_id']
from_vault(lr, Fake({'Return JSON: {"concepts"': dict(concepts=[dict(name='Softmax', statement='', quote='Softmax gives weights.', section='', ways=['words'])])}), d2)
test_eq(placement(lr.concept('softmax'), did).page, 4)
test_eq(placement(lr.concept('softmax'), d2).page, 1)
test_eq(lr.concept('softmax').ways, ['code', 'maths', 'words'])

A quote that is not in the text gives no passage. A figure file that is not there gives no figure.

In [ ]:
bad = dict(concepts=[dict(name='Masking', statement='The decoder hides later positions.', quote='we mask out all values', figure='images/nope.png', section='3.2.3')])
lr2 = temp_learning(); d3 = add_paper(lr2); from_vault(lr2, Fake({'Return JSON: {"concepts"': bad}), d3)
m = placement(lr2.concept('masking'), d3)
test_eq((m.passage, m.span, m.figure), ('', None, None))

A long paper has 15 pages. The model sees the paper in windows, and the last idea still gets its page.

In [ ]:
long = [(i, f'{i + 1} Section {i + 1}\n\n' + f'Idea {i + 1} is stated here. ' * 140) for i in range(15)]
lr3 = temp_learning(); d4 = lr3.vault.add(long, 'Long paper', source='/tmp/long.pdf', kind='pdf')['doc_id']
def per_window(p, sp):
    ns = sorted({int(n) for n in re.findall(r'Idea (\d+) is stated', p)})
    return json.dumps(dict(concepts=[dict(name=f'Idea {n}', statement=f'Idea {n}.', quote=f'Idea {n} is stated here.', section=f'{n} Section {n}') for n in ns]))
from_vault(lr3, per_window, d4)
test_eq(placement(lr3.concept('idea-15'), d4).page, 15)

`locate` folds white space and case. It refuses a quote that is too short.

In [ ]:
pg = [(1, 'One two.\n\nThree   FOUR five six.'), (2, 'Seven eight nine ten.')]
test_eq(locate(pg, 'three four FIVE six'), (1, 10, 31))
test_eq(locate(pg, 'seven eight nine'), (2, 0, 16))
test_is(locate(pg, 'three'), None)
test_is(locate(pg, 'nothing of this is there'), None)

`figure_path` keeps a figure only when the file is there, stays under the assets folder and has an image suffix.

In [ ]:
test_eq(figure_path(lr, '/tmp/attention.pdf', 'images/fig2.png'), 'attention/images/fig2.png')
test_is(figure_path(lr, '/tmp/attention.pdf', '../../../etc/hosts'), None)
test_is(figure_path(lr, '/tmp/attention.pdf', '/etc/hosts'), None)
(lr.vault.assets('attention')/'images'/'x.txt').write_text('x')
test_is(figure_path(lr, '/tmp/attention.pdf', 'images/x.txt'), None)

## vedicreader

The import reads the files of the vedicreader repo. It does not read the website. `$VEDICREADER_ROOT` gives the path of the repo.

| source | concepts |
|---|---|
| Sanskrit lessons | One concept for each lesson. The lesson before it in the course is its prereq. Its `check` pairs become ready questions at depth 4 (apply). Its recap becomes the statement. |
| `static/grammar/terms/*.json` | One concept for each term, with `related` links. A term with a jyotiṣa term of the same name also links to it. |
| `vr/jyotisha/explain/*.json` | One concept for each graha, rāśi and nakṣatra. |

`VR_ROOT` is the repo path. `plain` removes the marks of vedicreader from a text.

In [ ]:
#| export
VR_ROOT = Path(os.getenv('VEDICREADER_ROOT', Path.home()/'code/personal/orgs/vedicreader'))

def plain(s:str) -> str:
    "The vedicreader text without its `{…}` marks and `[1,2]` citations."
    return re.sub(r'\[\d+(?:,\s*\d+)*\]', '', re.sub(r'\{([^{}]*)\}', r'\1', s or '')).strip()

Three generators read one source each. Each generator yields the concepts as dicts for `put`. `_lesson_concepts` reads the lessons.

In [ ]:
#| export
def _lesson_concepts(root):
    d = root/'static/learn'
    order, prev = json.loads((d/'order.json').read_text()), None
    for ui, u in enumerate(order['units']):
        for li, ls in enumerate(u['lessons']):
            p = d/'lessons'/f"{ls['slug']}.json"
            body = json.loads(p.read_text()).get('body', []) if p.exists() else []
            recap = [plain(x) for b in body for x in b.get('recap', [])]
            qs = [dict(q=plain(c['q']), a=plain(c['a']), depth=4) for b in body for c in b.get('check', [])]
            yield dict(name=ls['title'], statement=' '.join([plain(ls['cap']), *recap]), subject='sanskrit', slug=f"sa-{ls['slug']}",
                       prereqs=[prev] if prev else [], questions=qs, order=ui*100 + li, unit=u['name'],
                       provenance=[dict(src='vedicreader', lesson=ls['slug'], url=f"https://vedicreader.com/v/learn/{ls['slug']}")])
            prev = f"sa-{ls['slug']}"

`_term_concepts` reads the grammar terms.

In [ ]:
#| export
def _term_concepts(root):
    for p in sorted((root/'static/grammar/terms').glob('*.json')):
        for t in json.loads(p.read_text()).get('terms', []):
            en = ', '.join(t.get('en') or [])
            yield dict(name=t.get('iast') or t['slug'], statement=plain(t.get('defn', '')), subject='sanskrit', slug=f"sa-term-{t['slug']}",
                       aliases=[a for a in (t.get('sa'), *(t.get('en') or ())) if a], group=t.get('group', ''), order=1000,
                       related=[f"sa-term-{r}" for r in t.get('related') or ()] + ([f"jy-term-{t['jy']}"] if t.get('jy') else []),
                       provenance=[dict(src='vedicreader', term=t['slug'], cite=(t.get('src') or {}).get('cite', ''))],
                       questions=[dict(q=f"What is {t.get('iast') or t['slug']} ({en})? Say what it is and where it is used.",
                                       a=plain(t.get('defn', '')), depth=2)] if t.get('defn') else [])

`_explain_concepts` reads the jyotiṣa explanations. The rāśi file keys each sign by its number, so `RASI` gives the names.

In [ ]:
#| export
RASI = [('Meṣa', 'Aries'), ('Vṛṣabha', 'Taurus'), ('Mithuna', 'Gemini'), ('Karkaṭa', 'Cancer'), ('Siṃha', 'Leo'), ('Kanyā', 'Virgo'),
        ('Tulā', 'Libra'), ('Vṛścika', 'Scorpio'), ('Dhanus', 'Sagittarius'), ('Makara', 'Capricorn'), ('Kumbha', 'Aquarius'), ('Mīna', 'Pisces')]

def _explain_name(kind, key):
    "The name and aliases of one explained item; the rāśi file keys each sign by its number, 0 for Meṣa."
    if kind == 'rasi' and key.isdigit(): nm, en = RASI[int(key)]; return nm, [en, 'rāśi', 'sign']
    return key, [kind]

def _explain_concepts(root):
    for kind in ('graha', 'rasi', 'nakshatra'):
        p = root/f'vr/jyotisha/explain/{kind}.json'
        if not p.exists(): continue
        for name, v in json.loads(p.read_text()).items():
            secs = v.get('sections', {})
            first_sec = next(iter(secs.values()), [])
            st = plain(first_sec[0] if first_sec else '')[:900]
            nm, al = _explain_name(kind, name)
            yield dict(name=nm, statement=st, subject='jyotisha', slug=f"jy-{kind}-{slugify(nm)}", kind=kind, order=2000,
                       aliases=al, provenance=[dict(src='vedicreader', explain=kind, key=name)])

`from_vedicreader` runs the generators that `parts` names and returns a count for each part.

In [ ]:
#| export
def from_vedicreader(lr:Learning, root=None, parts=('lessons', 'terms', 'explain')) -> dict:
    "Add the lessons, terms and jyotiṣa explanations of vedicreader as concepts. Keep the learner state on the shelf."
    root, n = Path(root or VR_ROOT), {}
    gens = dict(lessons=_lesson_concepts, terms=_term_concepts, explain=_explain_concepts)
    for part in parts:
        n[part] = 0
        for c in gens[part](root):
            lr.put(**c); n[part] += 1
    return n

`plain` removes the marks and the citations.

In [ ]:
test_eq(plain('{a} + {a} → {ā}: {nāsti}.[1,2]'), 'a + a → ā: nāsti.')

When the repo is on this machine, the lessons become a chain of concepts with prereqs and ready questions.

In [ ]:
if VR_ROOT.exists():
    lr = temp_learning()
    n = from_vedicreader(lr, parts=('lessons',))
    assert n['lessons'] > 10
    c = lr.concept('sa-sandhi-vowels')
    assert c.prereqs and c.questions and c.subject == 'sanskrit'
    test_eq(lr.concept('sa-sounds').prereqs, [])
    test_eq(_explain_name('rasi', '0'), ('Meṣa', ['Aries', 'rāśi', 'sign']))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()